In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.alerts import FlowDetector
from src.pipeline import DATA, RESULTS

det = FlowDetector(RESULTS, threshold=1.0)             # placeholder, replaced below
val = pd.read_parquet(DATA / "val.parquet").sample(200_000, random_state=11)
BUDGET = 0.05
det.threshold = float(np.quantile(det.score(val), 1 - BUDGET))
json.dump({"budget": BUDGET, "threshold": det.threshold, "model": "autoencoder-v2",
           "calibrated_on": "ToN-IoT benign validation split"},
          open(RESULTS / "alert_config.json", "w"), indent=2)
print("threshold:", det.threshold)


threshold: 0.005736944265663624


In [2]:
tune = pd.read_parquet(DATA / "tune.parquet").sample(50_000, random_state=11).sort_values("row_id")
n_alerts, hits = 0, 0
with open(RESULTS / "alerts_demo.jsonl", "w") as f:
    for i in range(0, len(tune), 5_000):                # small batches, like live traffic
        batch = tune.iloc[i:i + 5_000]
        truth = batch.set_index("row_id")["Attack"]
        for a in det.alerts(batch, id_prefix="tune"):
            a["_replay_truth"] = str(truth[a["flow_seq"]])   # only exists because the replay is labelled
            f.write(json.dumps(a) + "\n")
            n_alerts += 1
            hits += a["_replay_truth"] != "Benign"

n_benign = int((tune["Attack"] == "Benign").sum())
n_attack = len(tune) - n_benign
print(f"flows: {len(tune)} | alerts: {n_alerts}")
print(f"attack flows alerted: {hits}/{n_attack} ({hits / n_attack:.1%})")
print(f"benign flows alerted: {n_alerts - hits}/{n_benign} ({(n_alerts - hits) / n_benign:.1%})")
print(json.dumps(json.loads(open(RESULTS / "alerts_demo.jsonl").readline()), indent=2))

flows: 50000 | alerts: 22481
attack flows alerted: 20108/21588 (93.1%)
benign flows alerted: 2373/28412 (8.4%)
{
  "alert_id": "tune-7766125",
  "emitted_at": "2026-11-15T21:05:37+00:00",
  "flow_seq": 7766125,
  "src_ip": "192.168.1.32",
  "dst_ip": "192.168.1.1",
  "dst_port": 53,
  "protocol": 17,
  "score": 0.00852,
  "threshold": 0.00574,
  "severity": "low",
  "top_features": [
    "DNS_QUERY_TYPE",
    "DNS_QUERY_ID",
    "L7_PROTO"
  ],
  "model": "autoencoder-v2",
  "_replay_truth": "xss"
}


In [3]:
s = det.score(tune)
val_s = det.score(val)
is_atk = (tune["Attack"] != "Benign").to_numpy()

rows = []
for b in [0.001, 0.01, 0.05]:
    thr = np.quantile(val_s, 1 - b)
    flagged = s > thr
    rows.append({"budget": b, "threshold": round(float(thr), 5),
                 "attack_flows_alerted": flagged[is_atk].mean(),
                 "benign_flows_alerted": flagged[~is_atk].mean(),
                 "alerts": int(flagged.sum())})
pd.DataFrame(rows).round(4)

,budget,threshold,attack_flows_alerted,benign_flows_alerted,alerts
0,0.001,0.2264,0.0004,0.0007,30
1,0.010,0.0496,0.0069,0.0125,506
2,0.050,0.0057,0.9314,0.0835,22481


In [4]:
alerts = pd.read_json(RESULTS / "alerts_demo.jsonl", lines=True)
out = pd.DataFrame({"alerted": alerts["_replay_truth"].value_counts(),
                    "total": tune["Attack"].value_counts()}).fillna(0)
out["rate"] = out["alerted"] / out["total"]
out.round(3)

,alerted,total,rate
Benign,2373,28412,0.084
ddos,234,246,0.951
dos,120,130,0.923
injection,250,259,0.965
mitm,4,4,1.0
password,402,427,0.941
scanning,19010,20416,0.931
xss,88,106,0.83


In [5]:
rows = []
for b in [0.01, 0.015, 0.02, 0.03, 0.04, 0.05]:
    thr = np.quantile(val_s, 1 - b)          # threshold still comes from benign validation data
    f = s > thr
    rows.append({"budget": b, "attack_alerted": f[is_atk].mean(),
                 "benign_alerted": f[~is_atk].mean(), "alerts": int(f.sum())})
pd.DataFrame(rows).round(4)

,budget,attack_alerted,benign_alerted,alerts
0,0.010,0.0069,0.0125,506
1,0.015,0.0160,0.0237,1019
2,0.020,0.4355,0.0305,10269
3,0.030,0.6820,0.0491,16118
4,0.040,0.8253,0.0664,19704
5,0.050,0.9314,0.0835,22481


In [6]:
qs = [0.5, 0.9, 0.95, 0.98, 0.985, 0.99]
tv = tune["Attack"].to_numpy()
rows = {"benign val": np.quantile(val_s, qs)}
for t in ["scanning", "ddos", "dos", "injection", "password", "xss"]:
    rows[t] = np.quantile(s[tv == t], qs)
print("score quantiles (threshold at 1% budget = ", round(float(np.quantile(val_s, 0.99)), 4), ")")
pd.DataFrame(rows, index=qs).T.round(4)

score quantiles (threshold at 1% budget =  0.0496 )


,0.500,0.900,0.950,0.980,0.985,0.990
benign val,0.0001,0.0011,0.0057,0.0244,0.0332,0.0496
scanning,0.0207,0.0301,0.0305,0.0307,0.0307,0.0308
ddos,0.0122,0.0406,0.0442,0.0444,0.0482,0.0547
dos,0.0136,0.0462,0.0633,0.0785,0.0842,0.0847
injection,0.0180,0.0788,0.1027,0.1757,0.2118,0.2366
password,0.0109,0.0295,0.0295,0.0642,0.0817,0.0905
xss,0.0101,0.0532,0.0629,0.1089,0.1173,0.1268
